In [3]:
#configuration, edit accordingly between runs
#if change DENSE_WEIGHT, BM25_WEIGHT, or FIRST_STAGE, delete the cache file and restart the kernel
#RERANKER is in the cache filename, so changing it starts a fresh cache automatically
#if change DECOMP_PROMPT or DECOMP_MODEL, delete ../data/processed/decomp_cache.pkl

VERSION = "v2"          # corpus: v1 = current, v2 = serialization, v3 = + contextual
GOLD_VERSION = "v3"     # gold revision, independent of the corpus version.
                        # v3 = audited minimal gold_chunks over v2 corpus indices
                        # (v2 over-specified split-table refs, depressing all_gold@k)

CHUNKS_PATH = f"../data/processed/all_chunks_{VERSION}.pkl"
EMB_PATH    = f"../data/processed/embeddings_{VERSION}.npy"
RERANKER = "BAAI/bge-reranker-v2-m3"   # XLM-R large, 8194 max position embeddings
RERANK_MAX_LENGTH = 1024               # covers query + passage; base was capped at 512

# the rerank cache stores scores from one specific reranker, keyed by query text
# alone, so the model name is part of the filename. without that, switching
# rerankers silently serves the old model's scores under the new model's label.
CACHE_PATH  = f"../data/processed/rerank_cache_{VERSION}_{RERANKER.split('/')[-1]}.pkl"
GOLD_PATH   = f"../data/gold/gold_set_60_{GOLD_VERSION}.jsonl"
RESULTS_DIR = "../results"

DENSE_WEIGHT, BM25_WEIGHT = 3.0, 1.0
FIRST_STAGE = 50

In [1]:
import json
import pickle

src = "../data/gold/all_chunks_v2.jsonl"
dst = "../data/processed/all_chunks_v2.pkl"

with open(src, "r", encoding="utf-8") as f:
    v2_chunks = [json.loads(line) for line in f if line.strip()]

assert len(v2_chunks) == 3996

with open(dst, "wb") as f:
    pickle.dump(v2_chunks, f)

print("saved:", len(v2_chunks))
print(v2_chunks[0]["chunk_id"])
print(v2_chunks[-1]["chunk_id"])

saved: 3996
aapl_FY2025_10K:4:1
pltr_FY2025_10K:1537:1


In [2]:
import pickle
import numpy as np
from sentence_transformers import SentenceTransformer

with open("../data/processed/all_chunks_v2.pkl", "rb") as f:
    v2_chunks = pickle.load(f)

assert len(v2_chunks) == 3996

model = SentenceTransformer("BAAI/bge-small-en-v1.5")

texts = [c["text"] for c in v2_chunks]

v2_embeddings = model.encode(
    texts,
    normalize_embeddings=True,
    show_progress_bar=True,
)

np.save("../data/processed/embeddings_v2.npy", v2_embeddings)

print(v2_embeddings.shape)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/125 [00:00<?, ?it/s]

(3996, 384)


In [4]:
#load corpus, build bm25 and load gold
import json, pickle, os, glob
import numpy as np
import pandas as pd
from collections import defaultdict
from sentence_transformers import SentenceTransformer, CrossEncoder
from rank_bm25 import BM25Okapi

os.makedirs(RESULTS_DIR, exist_ok=True)

with open(CHUNKS_PATH, "rb") as f:
    all_chunks = pickle.load(f)
embeddings = np.load(EMB_PATH)

assert len(all_chunks) == embeddings.shape[0], \
    f"corpus/embeddings mismatch: {len(all_chunks)} vs {embeddings.shape[0]} — re-embed"

model = SentenceTransformer("BAAI/bge-small-en-v1.5")
bm25 = BM25Okapi([c["text"].lower().split() for c in all_chunks])

gold = [json.loads(l) for l in open(GOLD_PATH, encoding="utf-8") if l.strip()]

print(f"{VERSION}: {len(all_chunks)} chunks, {embeddings.shape}, {len(gold)} questions")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

v2: 3996 chunks, (3996, 384), 60 questions


In [5]:
#retrieval process with dense and bm25 search. query vector and seach pool cached
_qvec, _pool = {}, {}
RRF_K = 60

def encode_query(query):
    if query not in _qvec:
        _qvec[query] = model.encode([query], normalize_embeddings=True)[0]
    return _qvec[query]

def dense_search(query, k=FIRST_STAGE):
    return list(np.argsort(embeddings @ encode_query(query))[::-1][:k])

def bm25_search(query, k=FIRST_STAGE):
    return list(np.argsort(bm25.get_scores(query.lower().split()))[::-1][:k])

RRF_K = 60
def rrf_weighted(rankings_weights, k=RRF_K):
    scores = defaultdict(float)
    for ranking, weight in rankings_weights:
        for rank, doc_id in enumerate(ranking):
            scores[doc_id] += weight / (k + rank + 1)
    return sorted(scores, key=scores.get, reverse=True)

def fused_pool(query, n=FIRST_STAGE):
    if query not in _pool:
        _pool[query] = rrf_weighted([
            (dense_search(query, n), DENSE_WEIGHT),
            (bm25_search(query, n), BM25_WEIGHT),
        ])
    return _pool[query]

def hybrid_search(query, k=10):
    return fused_pool(query)[:k]

In [6]:
#reranker, lazy loaded and disk cached to save compute.
#bge-reranker-base is XLM-R base with a hard 512 ceiling covering query AND passage.
#XLM-R tokenizes this corpus at ~1.15x the BGE count, up to ~1.70x on number-dense
#tables, so 230/3996 chunks were silently truncated - worst on exactly the table
#chunks the serialization work targets. v2-m3 (XLM-R large, 8194 positions) at
#max_length=1024 clears that ceiling before contextualization adds 60-115 more
#XLM-R tokens per chunk. note: ~568M params vs ~278M, so expect a slower warm-up.
_reranker = None
_rerank_cache = {}

if os.path.exists(CACHE_PATH):
    with open(CACHE_PATH, "rb") as f:
        _rerank_cache = pickle.load(f)
    print(f"loaded {len(_rerank_cache)} cached rerank scores")

def get_reranker():
    global _reranker
    if _reranker is None:
        print("loading cross-encoder...")
        _reranker = CrossEncoder(RERANKER, max_length=RERANK_MAX_LENGTH)
    return _reranker

def _rerank_candidates(query):
    if query not in _rerank_cache:
        candidates = fused_pool(query)[:FIRST_STAGE]
        pairs = [(query, all_chunks[i]["text"]) for i in candidates]
        raw = np.asarray(
            get_reranker().predict(pairs, batch_size=len(pairs)),
            dtype=float,
        )
        _rerank_cache[query] = (list(candidates), raw)
    return _rerank_cache[query]

def search_reranked(query, k=10):
    candidates, raw = _rerank_candidates(query)
    return [candidates[i] for i in np.argsort(raw)[::-1][:k]]

In [ ]:
#5b query decomposition. the LLM call is cached to disk keyed on question text alone,
#so the cache is corpus independent and survives re-chunking.
#note: anthropic 1.0.0 removed sampling params - Messages.create takes no temperature,
#so determinism across runs comes from the disk cache, not from the decoding config.
#delete decomp_cache.pkl if you change DECOMP_PROMPT or DECOMP_MODEL.
from anthropic import Anthropic
from dotenv import load_dotenv

load_dotenv("../.env")
_client = Anthropic()

DECOMP_CACHE = "../data/processed/decomp_cache.pkl"
DECOMP_MODEL = "claude-haiku-4-5-20251001"

_decomp_cache = {}
if os.path.exists(DECOMP_CACHE):
    with open(DECOMP_CACHE, "rb") as f:
        _decomp_cache = pickle.load(f)
    print(f"loaded {len(_decomp_cache)} cached decompositions")

DECOMP_PROMPT = """You are preparing search queries for a retrieval system over SEC 10-K filings.

Question: {question}

If answering this requires facts from two or more separate places - different companies,
different fiscal years, or different sections - split it into independent sub-queries.
Each sub-query must be self-contained and name its company and fiscal period explicitly.

If the question asks for a single fact from one place, return it unchanged as a single item.

Output only a JSON array of strings. No preamble, no markdown fences."""


def _strip_fences(raw):
    raw = raw.strip()
    if raw.startswith("```"):
        raw = raw.split("\n", 1)[1] if "\n" in raw else raw[3:]
        if raw.rstrip().endswith("```"):
            raw = raw.rstrip()[:-3]
    return raw.strip()


def decompose(question):
    if question in _decomp_cache:
        return _decomp_cache[question]

    resp = _client.messages.create(
        model=DECOMP_MODEL,
        max_tokens=300,
        messages=[{"role": "user", "content": DECOMP_PROMPT.format(question=question)}],
    )

    try:
        subs = json.loads(_strip_fences(resp.content[0].text))
        ok = (isinstance(subs, list) and subs
              and all(isinstance(s, str) and s.strip() for s in subs))
        if not ok:
            subs = [question]
    except json.JSONDecodeError:
        subs = [question]

    _decomp_cache[question] = subs
    return subs

In [ ]:
#5c decomposed search. round robin, not concatenation: with two sub-queries at k=10
#each contributes roughly its top 5, so one company cannot occupy every slot.
#that monopolisation is the exact failure this config exists to fix.
def search_decomposed(query, k=10):
    subs = decompose(query)

    # DECOMP_PROMPT says to return an atomic question unchanged, but the model
    # still paraphrases ("What was" -> "What were"). Reranking a paraphrase would
    # make this config differ from the rerank baseline on questions that never
    # decomposed - a confound. Use the original so the only difference between
    # `rerank` and `decomposed` is the questions that actually split.
    if len(subs) == 1:
        return search_reranked(query, k)

    per_sub = [search_reranked(s, k) for s in subs]

    merged, seen = [], set()
    for rank in range(k):
        for lst in per_sub:
            if rank < len(lst) and lst[rank] not in seen:
                seen.add(lst[rank])
                merged.append(lst[rank])
                if len(merged) == k:
                    return merged

    # heavy overlap between sub-queries can leave the merge short of k. top up from
    # the undecomposed query so every config is scored on the same number of results.
    if len(merged) < k:
        for i in search_reranked(query, k):
            if i not in seen:
                seen.add(i)
                merged.append(i)
                if len(merged) == k:
                    break

    return merged

In [ ]:
#5d sanity check the split before spending on 60 questions (2 API calls).
#expect two sub-queries for the first and one unchanged for the second.
#if atomic questions are being split, tighten DECOMP_PROMPT before warming caches.
for _q in [
    "Between Meta's fiscal year 2025 and NVIDIA's fiscal year 2026, which company reported higher total revenue?",
    "What was Apple's total net sales for fiscal year 2025?",
]:
    print(_q)
    for _s in decompose(_q):
        print("   ->", _s)
    print()

In [7]:
#9 DECOMPOSE FIRST, then price the rerank job before committing to it.
#decomposition is cheap and disk cached; reranking is ~71s/query on CPU with
#bge-reranker-v2-m3. running it in this order means a broken prompt or an
#over-split question shows up here instead of 70 minutes into a warm-up.
from collections import Counter

for i, q in enumerate(gold):
    decompose(q["question"])
    print(f"decompose {i+1}/{len(gold)}", end="\r")

with open(DECOMP_CACHE, "wb") as f:
    pickle.dump(_decomp_cache, f)

answerable = [q for q in gold if q["gold_chunks"]]
print(f"\n{len(gold)} questions, {len(answerable)} answerable")
print("sub-queries per question:",
      dict(sorted(Counter(len(decompose(q["question"])) for q in gold).items())))

#evaluate() only ever searches answerable questions, so the 20 unanswerable ones
#and their sub-queries never need warm scores. warming them was 67 wasted minutes.
to_rerank = {q["question"] for q in answerable}
for q in answerable:
    subs = decompose(q["question"])
    if len(subs) > 1:
        to_rerank.update(subs)

to_rerank = sorted(to_rerank)
missing = [s for s in to_rerank if s not in _rerank_cache]

print(f"\nevaluate() needs {len(to_rerank)} queries reranked "
      f"({len(to_rerank) - len(missing)} already cached, {len(missing)} to compute)")
print(f"estimated cell 10 runtime: {len(missing) * 71 / 60:.0f} min")

#round robin gives each sub-query only ~k/n slots, so 3+ subs is usually a
#prompt failure rather than a genuinely multi-part question. inspect these.
print("\nquestions producing 3+ sub-queries:")
for q in gold:
    subs = decompose(q["question"])
    if len(subs) >= 3:
        print(f"  {len(subs)} subs  [{q['id']}] answerable={bool(q['gold_chunks'])}  {q['question'][:60]}")
        for s in subs:
            print(f"        - {s}")

loading cross-encoder...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

60/60
cached 60 questions → ../data/processed/rerank_cache_v2.pkl


In [ ]:
#10 one rerank pass over exactly what evaluate() will ask for. check the count
#and estimate printed by cell 9 before starting this - on CPU it is the long pole.
import time

t0 = time.time()
for i, s in enumerate(missing):
    _rerank_candidates(s)
    print(f"rerank {i+1}/{len(missing)}   {(time.time()-t0)/60:5.1f} min elapsed", end="\r")

with open(CACHE_PATH, "wb") as f:
    pickle.dump(_rerank_cache, f)

print(f"\ncached {len(_rerank_cache)} queries -> {CACHE_PATH}")
print(f"took {(time.time()-t0)/60:.1f} min")

In [8]:
#metrics to evaluate recall k
def evaluate(search_fn, ks=(1, 5, 10, 20)):
    answerable = [q for q in gold if q["gold_chunks"]]
    hard       = [q for q in gold if q["type"] == "hard" and q["gold_chunks"]]
    easy       = [q for q in gold if q["type"] == "easy" and q["gold_chunks"]]

    full = {q["id"]: search_fn(q["question"], max(ks)) for q in answerable}

    m = {"n_answerable": len(answerable), "n_hard": len(hard), "n_easy": len(easy)}

    for k in ks:
        ret = {qid: set(r[:k]) for qid, r in full.items()}

        for label, subset in (("", answerable), ("easy_", easy), ("hard_", hard)):
            hits = sum(1 for q in subset if ret[q["id"]] & set(q["gold_chunks"]))
            m[f"{label}recall@{k}"] = round(hits / len(subset), 3)
            m[f"{label}recall@{k}_hits"] = hits

        complete = sum(1 for q in hard if set(q["gold_chunks"]).issubset(ret[q["id"]]))
        m[f"hard_all_gold@{k}"] = round(complete / len(hard), 3)
        m[f"hard_all_gold@{k}_hits"] = complete

    return m


def run_and_save(search_fn, config_name, config_dict):
    metrics = evaluate(search_fn)
    payload = {
        "version": VERSION,
        "gold_version": GOLD_VERSION,
        "config_name": config_name,
        "n_chunks": len(all_chunks),
        "config": config_dict,
        "metrics": metrics,
    }
    path = f"{RESULTS_DIR}/{VERSION}_{config_name}.json"
    with open(path, "w") as f:
        json.dump(payload, f, indent=2)
    print(f"saved {path}")
    return payload

In [9]:
#running dense, bm25 and reranked search individually. remember to warm cache
run_and_save(dense_search, "dense",
             {"method": "dense", "model": "BAAI/bge-small-en-v1.5"})

run_and_save(hybrid_search, "hybrid",
             {"method": "weighted_rrf", "dense_weight": DENSE_WEIGHT,
              "bm25_weight": BM25_WEIGHT, "rrf_k": RRF_K})

run_and_save(search_reranked, "rerank",
             {"method": "weighted_rrf + cross-encoder",
              "reranker": RERANKER, "rerank_max_length": RERANK_MAX_LENGTH,
              "first_stage": FIRST_STAGE})

run_and_save(search_decomposed, "decomposed",
             {"method": "decompose + weighted_rrf + cross-encoder",
              "decomp_model": DECOMP_MODEL,
              "reranker": RERANKER,
              "rerank_max_length": RERANK_MAX_LENGTH,
              "first_stage": FIRST_STAGE})


saved ../results/v2_dense.json
saved ../results/v2_hybrid.json
saved ../results/v2_rerank.json


{'version': 'v2',
 'config_name': 'rerank',
 'n_chunks': 3996,
 'config': {'method': 'weighted_rrf + cross-encoder',
  'reranker': 'BAAI/bge-reranker-base',
  'first_stage': 50},
 'metrics': {'n_answerable': 40,
  'n_hard': 20,
  'n_easy': 20,
  'recall@1': 0.3,
  'recall@1_hits': 12,
  'easy_recall@1': 0.45,
  'easy_recall@1_hits': 9,
  'hard_recall@1': 0.15,
  'hard_recall@1_hits': 3,
  'hard_all_gold@1': 0.0,
  'hard_all_gold@1_hits': 0,
  'recall@5': 0.5,
  'recall@5_hits': 20,
  'easy_recall@5': 0.6,
  'easy_recall@5_hits': 12,
  'hard_recall@5': 0.4,
  'hard_recall@5_hits': 8,
  'hard_all_gold@5': 0.0,
  'hard_all_gold@5_hits': 0,
  'recall@10': 0.75,
  'recall@10_hits': 30,
  'easy_recall@10': 0.85,
  'easy_recall@10_hits': 17,
  'hard_recall@10': 0.65,
  'hard_recall@10_hits': 13,
  'hard_all_gold@10': 0.0,
  'hard_all_gold@10_hits': 0,
  'recall@20': 0.875,
  'recall@20_hits': 35,
  'easy_recall@20': 0.95,
  'easy_recall@20_hits': 19,
  'hard_recall@20': 0.8,
  'hard_recall@20

In [10]:
#comparison table
rows = []
for path in sorted(glob.glob(f"{RESULTS_DIR}/*.json")):
    r = json.load(open(path))
    if "config_name" not in r:
        continue
    m = r["metrics"]
    rows.append({
        "run": f"{r['version']}_{r['config_name']}",
        "chunks": r["n_chunks"],
        "r@1": m["recall@1"], "r@5": m["recall@5"],
        "r@10": m["recall@10"], "r@20": m["recall@20"],
        "easy@1": m["easy_recall@1"],
        "hard@10": m["hard_recall@10"],
        "allgold@10": m["hard_all_gold@10"],
        "allgold@20": m["hard_all_gold@20"],
    })

pd.DataFrame(rows).set_index("run")

,chunks,r@1,r@5,r@10,r@20,easy@1,hard@10,allgold@10,allgold@20
run,,,,,,,,,
v1_dense,3839,0.225,0.425,0.600,0.850,0.25,0.60,0.20,0.40
v1_hybrid,3839,0.225,0.425,0.625,0.800,0.25,0.65,0.15,0.35
v1_rerank,3839,0.300,0.450,0.625,0.775,0.45,0.55,0.00,0.25
v2_dense,3996,0.225,0.400,0.675,0.850,0.25,0.65,0.25,0.35
v2_hybrid,3996,0.225,0.425,0.650,0.875,0.25,0.60,0.15,0.40
v2_rerank,3996,0.300,0.500,0.750,0.875,0.45,0.65,0.00,0.30


In [ ]:
#freeze v1
import os
import numpy as np

print("v1 embeddings:", np.load("../data/processed/embeddings_v1.npy").shape)
print("v1 rerank cache:", os.path.exists("../data/processed/rerank_cache_v1.pkl"))

for name in ["v1_dense.json", "v1_hybrid.json", "v1_rerank.json"]:
    print(name, os.path.exists(f"../results/{name}"))

v1 embeddings: (3839, 384)
v1 rerank cache: True
v1_dense.json True
v1_hybrid.json True
v1_rerank.json True


In [11]:
#check failure chunks manually
def show_failures(search_fn, k=10, qtype=None, limit=5):
    subset = [q for q in gold if q["gold_chunks"]
              and (qtype is None or q["type"] == qtype)]
    shown = 0
    for q in subset:
        retrieved = search_fn(q["question"], k)
        if set(retrieved) & set(q["gold_chunks"]):
            continue
        print(f"FAIL [{q['id']}] {q['question']}")
        print(f"  gold: {q['gold_chunks']}")
        for i in retrieved[:3]:
            print(f"  got [{i}] {all_chunks[i]['doc']} | {all_chunks[i]['text'][:110]}")
        print()
        shown += 1
        if shown >= limit:
            break

In [13]:
def show_allgold_failures(search_fn, k=10, qtype="hard", limit=20):
    subset = [
        q for q in gold
        if q["gold_chunks"]
        and (qtype is None or q["type"] == qtype)
    ]

    shown = 0

    for q in subset:
        retrieved = search_fn(q["question"], k)

        gold_set = set(q["gold_chunks"])
        retrieved_set = set(retrieved)

        if gold_set.issubset(retrieved_set):
            continue

        missing = gold_set - retrieved_set

        print(f"FAIL [{q['id']}] {q['question']}")
        print(f"  gold:      {q['gold_chunks']}")
        print(f"  missing:   {sorted(missing)}")
        print(f"  retrieved: {retrieved[:10]}")
        print()

        shown += 1
        if shown >= limit:
            break

In [14]:
show_allgold_failures(
    search_reranked,
    k=10,
    qtype="hard",
    limit=20,
)

FAIL [q021] How much did NVIDIA's total revenue grow in dollar terms from fiscal year 2024 to fiscal year 2026?
  gold:      [2558, 3308]
  missing:   [2558, 3308]
  retrieved: [np.int64(3485), np.int64(3081), np.int64(3445), np.int64(3361), np.int64(3299), np.int64(3487), np.int64(3078), np.int64(2545), np.int64(3481), np.int64(3486)]

FAIL [q022] How did NVIDIA's total employee headcount change from the end of fiscal year 2024 to the end of fiscal year 2026?
  gold:      [2436, 3196]
  missing:   [2436]
  retrieved: [np.int64(3485), np.int64(3196), np.int64(3481), np.int64(2544), np.int64(3298), np.int64(2555), np.int64(3443), np.int64(3441), np.int64(3315), np.int64(3445)]

FAIL [q023] Between Meta's fiscal year 2025 and NVIDIA's fiscal year 2026, which company reported higher total revenue?
  gold:      [1772, 3308]
  missing:   [1772]
  retrieved: [np.int64(3485), np.int64(3445), np.int64(3299), np.int64(3298), np.int64(3441), np.int64(3177), np.int64(3444), np.int64(3487), np.int

In [15]:
#inspecting near misses from adjacent chunks for q32
#did the reranker actually retrieve the wrong evidence, or merely a neighboring chunk that contains equivalent evidence?
q = next(q for q in gold if q["id"] == "q032")

print("GOLD")
for i in q["gold_chunks"]:
    print(f"\n--- {i} ---")
    print(all_chunks[i]["text"])

print("\nRETRIEVED NEAR-MISSES")
for i in [3793, 3795]:
    print(f"\n--- {i} ---")
    print(all_chunks[i]["text"])

GOLD

--- 3761 ---
PALANTIR 10-K FY2025 | FY2025 | Part II > Item 7 — MANAGEMENT’S DISCUSSION AND ANALYSIS OF FINANCIAL CONDITION AND RESULTS OF OPERATIONS > Total Remaining Deal Value

We are focused on building strategic relationships with, and delivering significant outcomes for, our customers over the long term. Our contracts with our customers reflect that long-term orientation, often lasting for multiple years at a time.

Total remaining deal value is the total remaining value, as of the end of the reporting period, of contracts that have been entered into with, or awarded by, our customers. Total remaining deal value presumes the exercise of all contract options available to our customers and no termination of contracts. However, many of our contracts are subject to termination provisions, including for convenience, and there can be no guarantee that contracts are not terminated or that contract options will be exercised. Further, total remaining deal value may exclude all or so

In [16]:
#test q40
q = next(q for q in gold if q["id"] == "q040")

print("QUESTION")
print(q["question"])

print("\nGOLD")
for i in q["gold_chunks"]:
    print(f"\n--- {i} ---")
    print(all_chunks[i]["text"])

print("\nRETRIEVED")
retrieved = search_reranked(q["question"], k=10)

for i in retrieved:
    print(f"\n--- {i} ---")
    print(all_chunks[i]["text"])

QUESTION
Which company had a higher gross margin percentage in its most recent fiscal year: Apple (FY2025) or Dell (FY2026)?

GOLD

--- 116 ---
APPLE 10-K FY2025 | FY2025 | Part II > Item 8 — Financial Statements and Supplementary Data > (In millions, except number of shares, which are reflected in thousands, and per-share amounts)

Years ended Net sales:
Columns: September 27, 2025 | September 28, 2024 | September 30, 2023
Products | September 27, 2025: $307,003 | September 28, 2024: $294,866 | September 30, 2023: $298,085
Services | September 27, 2025: 109,158 | September 28, 2024: 96,169 | September 30, 2023: 85,200
Total net sales | September 27, 2025: 416,161 | September 28, 2024: 391,035 | September 30, 2023: 383,285
Cost of sales:
Products | September 27, 2025: 194,116 | September 28, 2024: 185,233 | September 30, 2023: 189,282
Services | September 27, 2025: 26,844 | September 28, 2024: 25,119 | September 30, 2023: 24,855
Total cost of sales | September 27, 2025: 220,960 | Septe

In [17]:
from transformers import AutoTokenizer

bge_tok = AutoTokenizer.from_pretrained("BAAI/bge-small-en-v1.5")
xlm_tok = AutoTokenizer.from_pretrained("BAAI/bge-reranker-base")

q = "How much did NVIDIA's total revenue grow in dollar terms from fiscal year 2024 to fiscal year 2026?"
qn = len(xlm_tok.encode(q, add_special_tokens=False))

rows = []
for c in all_chunks:
    n_bge = len(bge_tok.encode(c["text"], add_special_tokens=False))
    n_xlm = len(xlm_tok.encode(c["text"], add_special_tokens=False))
    rows.append((c["chunk_id"], c.get("element_type"), n_bge, n_xlm, qn + n_xlm + 3))

import pandas as pd
df = pd.DataFrame(rows, columns=["chunk_id", "type", "bge", "xlm", "total_with_query"])
print(f"query = {qn} XLM tokens")
print(f"ratio xlm/bge: mean {(df.xlm/df.bge).mean():.2f}, max {(df.xlm/df.bge).max():.2f}")
print(f"chunks truncated at 512: {(df.total_with_query > 512).sum()} / {len(df)}")
print(df[df.total_with_query > 512].groupby("type").size())

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (596 > 512). Running this sequence through the model will result in indexing errors


query = 23 XLM tokens
ratio xlm/bge: mean 1.15, max 1.70
chunks truncated at 512: 230 / 3996
Series([], dtype: int64)


In [18]:
gold_ids = {i for q in gold for i in q["gold_chunks"]}
trunc = set(df[df.total_with_query > 512].index)
print(f"gold chunks truncated: {len(gold_ids & trunc)} / {len(gold_ids)}")

gold chunks truncated: 2 / 39


In [19]:
gold_ids = {i for q in gold for i in q["gold_chunks"]}
for i in sorted(gold_ids & set(df[df.total_with_query > 512].index)):
    qs = [q["id"] for q in gold if i in q["gold_chunks"]]
    print(i, qs, df.loc[i, "total_with_query"], all_chunks[i]["chunk_id"])

381 ['q004', 'q024', 'q035'] 574 amzn_FY2025_10K:456:1
2199 ['q013', 'q030'] 610 msft_FY2026_10K:583:1
